In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
    StandardScaler,
    MinMaxScaler
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv("../data/raw/application_train.csv")

In [4]:
df.shape

(307511, 122)

In [5]:
X = df.drop("TARGET", axis=1)

y = df["TARGET"]

In [6]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 121)
y shape: (307511,)


In [7]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

In [8]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21164\3327125511.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [9]:
print("Numerical:", len(numerical_cols))
print("Categorical:", len(categorical_cols))

Numerical: 105
Categorical: 16


In [10]:
missing_percent = (
    X.isnull().sum() / len(X)
) * 100

In [10]:
missing_percent.sort_values(
    ascending=False
).head(20)

COMMONAREA_AVG              69.872297
COMMONAREA_MEDI             69.872297
COMMONAREA_MODE             69.872297
NONLIVINGAPARTMENTS_AVG     69.432963
NONLIVINGAPARTMENTS_MEDI    69.432963
NONLIVINGAPARTMENTS_MODE    69.432963
FONDKAPREMONT_MODE          68.386172
LIVINGAPARTMENTS_MEDI       68.354953
LIVINGAPARTMENTS_AVG        68.354953
LIVINGAPARTMENTS_MODE       68.354953
FLOORSMIN_MODE              67.848630
FLOORSMIN_AVG               67.848630
FLOORSMIN_MEDI              67.848630
YEARS_BUILD_MODE            66.497784
YEARS_BUILD_MEDI            66.497784
YEARS_BUILD_AVG             66.497784
OWN_CAR_AGE                 65.990810
LANDAREA_MEDI               59.376738
LANDAREA_AVG                59.376738
LANDAREA_MODE               59.376738
dtype: float64

In [11]:
missing_df = pd.DataFrame({
    "Missing Percentage": missing_percent
})

missing_df = missing_df.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_df.head(20)

,Missing Percentage
COMMONAREA_AVG,69.872297
COMMONAREA_MEDI,69.872297
COMMONAREA_MODE,69.872297
NONLIVINGAPARTMENTS_AVG,69.432963
NONLIVINGAPARTMENTS_MEDI,69.432963
NONLIVINGAPARTMENTS_MODE,69.432963
FONDKAPREMONT_MODE,68.386172
LIVINGAPARTMENTS_MEDI,68.354953
LIVINGAPARTMENTS_AVG,68.354953
LIVINGAPARTMENTS_MODE,68.354953


## Missing Value Observations

- Several columns contain substantial missing data.
- Some housing-related features have more than 50% missing values.
- Missing values must be handled before model training.
- Columns with extremely high missingness may be removed later.

In [12]:
threshold = 60

In [13]:
missing_cols = missing_percent[
    missing_percent > threshold
].index

In [14]:
len(missing_cols)

17

In [15]:
missing_cols.tolist()

['OWN_CAR_AGE',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'FLOORSMIN_AVG',
 'LIVINGAPARTMENTS_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'YEARS_BUILD_MODE',
 'COMMONAREA_MODE',
 'FLOORSMIN_MODE',
 'LIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'YEARS_BUILD_MEDI',
 'COMMONAREA_MEDI',
 'FLOORSMIN_MEDI',
 'LIVINGAPARTMENTS_MEDI',
 'NONLIVINGAPARTMENTS_MEDI',
 'FONDKAPREMONT_MODE']

In [15]:
X = X.drop(columns=missing_cols)

In [16]:
X.shape

(307511, 104)

In [17]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = X.select_dtypes(
    include=["object"]
).columns

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21164\3481394536.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [18]:
print("Numerical:", len(numerical_cols))
print("Categorical:", len(categorical_cols))

Numerical: 89
Categorical: 15


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (246008, 104)
X_test: (61503, 104)
y_train: (246008,)
y_test: (61503,)


In [21]:
print(y_train.value_counts(normalize=True))

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [22]:
print(y_test.value_counts(normalize=True))

TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [24]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [25]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_cols
        ),
        (
            "cat",
            categorical_transformer,
            categorical_cols
        )
    ]
)

In [27]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [28]:
X_train_processed = preprocessor.fit_transform(
    X_train
)

In [30]:
X_test_processed = preprocessor.transform(
    X_test
)

In [31]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(246008, 225)
(61503, 225)


In [32]:
y_train.value_counts()

TARGET
0    226148
1     19860
Name: count, dtype: int64

In [33]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_train_balanced, y_train_balanced = ros.fit_resample(
    X_train_processed,
    y_train
)

In [34]:
pd.Series(y_train_balanced).value_counts()

TARGET
0    226148
1    226148
Name: count, dtype: int64

In [34]:
print(X_train_processed.shape)

(246008, 225)


In [35]:
y_train.value_counts()

TARGET
0    226148
1     19860
Name: count, dtype: int64

In [35]:
pd.Series(y_train_balanced).value_counts(normalize=True) * 100

TARGET
0    50.0
1    50.0
Name: proportion, dtype: float64

In [36]:
import joblib


In [37]:
joblib.dump(
    preprocessor,
    "../ml/preprocessor.pkl"
)

['../ml/preprocessor.pkl']

In [38]:
joblib.dump(
    X_train_balanced,
    "../data/processed/X_train.pkl"
)

joblib.dump(
    y_train_balanced,
    "../data/processed/y_train.pkl"
)

joblib.dump(
    X_test_processed,
    "../data/processed/X_test.pkl"
)

joblib.dump(
    y_test,
    "../data/processed/y_test.pkl"
)

['../data/processed/y_test.pkl']

In [39]:
print("X_train_balanced:", X_train_balanced.shape)
print("X_test_processed:", X_test_processed.shape)

print("y_train_balanced:", y_train_balanced.shape)
print("y_test:", y_test.shape)

X_train_balanced: (452296, 225)
X_test_processed: (61503, 225)
y_train_balanced: (452296,)
y_test: (61503,)
